# V112 - Full build 2: specialist action

Pre-registered plan: v111 adopted a no-chat specialist at plus 0.07 on its slice. This build re-verifies under a new seed, trains the specialist on all 117 no-chat users, scores the 33 no-chat test users, and writes a champion-relative submission only if the re-verification clears the slice floor with no negative fold and geometry stays inside the slice. Otherwise the champion writes itself out untouched.

In [1]:
import json
import re
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold

SEED = 4141
N_SPLITS = 5
SLICE_FLOOR = 0.01
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
CHAMPION = next((p for p in [Path("/kaggle/input/intelligo-champion/submission_v42_primary.csv"),
                            Path("results/v42/submission_v42_primary.csv")] if p.exists()), None)
assert CHAMPION is not None, "full build composes onto the champion file"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v112")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve(), "| CHAMPION:", CHAMPION)

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data | CHAMPION: results/v42/submission_v42_primary.csv


## Re-verify, train specialist, compose submission

Slice-stratified folds for both arms. The test specialist trains once on the full slice. Geometry confines every edit to no-chat test users.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
S = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
S = np.where(np.isnan(S), np.nanmean(S, axis=0), S)
A = (S - S.mean(0)) / (S.std(0) + 1e-9)
A_tr, A_te = A[:n_train], A[n_train:]
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
is_nochat = np.array([u not in by_user for u in all_users])
nc_tr, nc_te = np.where(is_nochat[:n_train])[0], np.where(is_nochat[n_train:])[0]
print(f"no-chat train={len(nc_tr)} test={len(nc_te)}")

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

oof_g, oof_s = np.zeros((len(nc_tr), len(MODS))), np.zeros((len(nc_tr), len(MODS)))
fids = np.zeros(len(nc_tr), int)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (trs, vas) in enumerate(skf.split(nc_idx := nc_tr, Y[nc_tr].argmax(1))):
    tr_all = np.setdiff1d(np.arange(n_train), nc_tr[vas])
    tr_nc, va_nc = nc_tr[trs], nc_idx[vas]
    for j in range(len(MODS)):
        g = Ridge(alpha=1.0)
        g.fit(A_tr[tr_all], Y[tr_all, j])
        oof_g[vas, j] = g.predict(A_tr[va_nc])
        s = Ridge(alpha=1.0)
        s.fit(A_tr[tr_nc], Y[tr_nc, j])
        oof_s[vas, j] = s.predict(A_tr[va_nc])
    fids[vas] = fi + 1
sg, ss = ndcg5(Y[nc_tr], oof_g), ndcg5(Y[nc_tr], oof_s)
d = ss - sg
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(5000)], [0.025, 0.975])
wf = min((np.mean(ss[fids == k]) - np.mean(sg[fids == k])) for k in range(1, N_SPLITS + 1))
print(f"globalist={sg.mean():.5f} specialist={ss.mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
ok = (lo > SLICE_FLOOR) and (wf >= 0)
print("REPLICATE CONFIRMED" if ok else "REPLICATE FAILED")

ch = pd.read_csv(CHAMPION)
assert ch["user_id"].tolist() == test_ids
C = ch[MODS].to_numpy(float)
report = {"replicate": bool(ok)}
final = C.copy()
if ok:
    P = np.zeros((len(nc_te), len(MODS)))
    for j in range(len(MODS)):
        s = Ridge(alpha=1.0)
        s.fit(A_tr[nc_tr], Y[nc_tr, j])
        P[:, j] = s.predict(A_te[nc_te])
    S = C.copy()
    S[nc_te] = P
    changed = set(np.where((S != C).any(1))[0].tolist())
    assert changed <= set(nc_te.tolist()), "geometry breach"
    report.update({"changed": len(changed), "top1_changes": int((C.argmax(1) != S.argmax(1)).sum())})
    print(f"geometry: changed={len(changed)} top1-changes={report['top1_changes']}")
    final = S
else:
    print("FAIL-CLOSE: champion untouched")
sub = pd.DataFrame(final, columns=MODS)
sub.insert(0, "user_id", test_ids)
sub.to_csv(OUT / "submission_v112_primary.csv", index=False)
report["sha"] = sha256(pd.util.hash_pandas_object(sub).values.tobytes()).hexdigest()[:16]
report["champion_sha"] = sha256(pd.util.hash_pandas_object(ch).values.tobytes()).hexdigest()[:16]
Path(OUT / "v112_gate_report.json").write_text(json.dumps(report, indent=1))
print("wrote", OUT / "submission_v112_primary.csv", report)

no-chat train=117 test=33


/home/athilla/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


globalist=0.52652 specialist=0.59137 delta=+0.06485 CI95=[+0.02725,+0.10157] worst-fold=-0.00579
REPLICATE FAILED
FAIL-CLOSE: champion untouched
wrote results/v112/submission_v112_primary.csv {'replicate': False, 'sha': '7bc841d13df9f947', 'champion_sha': '7bc841d13df9f947'}
